In [1]:
# Temporal validation on 2021-2023 (D-004, D-011, D-012, D-013).
# Run ONCE. Do not change the model, predictors or threshold after seeing these results.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

sys.path.insert(0, str(find_project_root() / "src"))
import importlib
import modeling as m
importlib.reload(m)

dev = m.load("2017-2020")
val = m.load("2021-2023")
y_dev = dev[m.OUTCOME].to_numpy()
y_val = val[m.OUTCOME].to_numpy()
w_val = val["exam_weight"].to_numpy()
print(f"Development 2017-2020: n = {len(dev)}, positive = {y_dev.mean():.1%}")
print(f"Validation  2021-2023: n = {len(val)}, positive = {y_val.mean():.1%}\n")

# ---- 1. Thresholds, fixed on development data only (D-013) ----
def cv_threshold(predictors):
    """Mean of the 10 per-repeat thresholds from out-of-fold predictions."""
    cv = m.cross_validate(m.make_logistic, predictors, dev)
    return cv["thresholds"]["threshold"].mean()

t_A = cv_threshold(m.MODEL_A)
t_B = cv_threshold(m.MODEL_B)
age_cutoff = m.threshold_for_sensitivity(y_dev, dev["age"].to_numpy())
print(f"Fixed thresholds from 2017-2020: Model A = {t_A:.3f}, Model B = {t_B:.3f}, "
      f"age rule = age >= {age_cutoff:.0f}\n")

# ---- 2. Fit on all development data, predict validation ----
model_A = m.make_logistic(m.MODEL_A).fit(dev[m.MODEL_A], y_dev)
model_B = m.make_logistic(m.MODEL_B).fit(dev[m.MODEL_B], y_dev)
p_A = model_A.predict_proba(val[m.MODEL_A])[:, 1]
p_B = model_B.predict_proba(val[m.MODEL_B])[:, 1]
age_val = val["age"].to_numpy()

candidates = {
    "Model A (final)": (p_A, t_A, True),
    "Model B": (p_B, t_B, True),
    "Age rule": (age_val, age_cutoff, False),   # not a probability: no calibration
}

# ---- 3. Performance: unweighted (primary) and weighted (secondary), D-012 ----
def evaluate(p, t, is_prob, w=None):
    out = {"roc_auc": m.roc_auc_score(y_val, p, sample_weight=w)}
    if is_prob:
        out.update(m.metrics(y_val, p, w))
    out.update(m.at_threshold(y_val, p, t, w))
    return out

COLS = ["roc_auc", "pr_auc", "brier", "cal_slope", "obs_exp",
        "sensitivity", "specificity", "precision", "referred_pct", "screen_per_case"]
for label, w in [("UNWEIGHTED (primary)", None), ("WEIGHTED, MEC exam weights (secondary)", w_val)]:
    table = pd.DataFrame({name: evaluate(p, t, is_prob, w)
                          for name, (p, t, is_prob) in candidates.items()}).T
    print(f"=== Validation 2021-2023, {label} ===")
    print(table.reindex(columns=COLS).round(3), "\n")

# ---- 4. Bootstrap 95% intervals for the final model (unweighted) ----
rng = np.random.default_rng(m.RANDOM_STATE)
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y_val), len(y_val))
    if y_val[i].min() == y_val[i].max():
        continue
    r = m.metrics(y_val[i], p_A[i])
    r.update(m.at_threshold(y_val[i], p_A[i], t_A))
    boot.append(r)
boot = pd.DataFrame(boot)
ci = boot[["roc_auc", "cal_slope", "obs_exp", "sensitivity", "specificity", "referred_pct"]] \
    .quantile([0.025, 0.975]).T
ci.columns = ["low", "high"]
print("=== Model A, bootstrap 95% interval (1000 resamples) ===")
print(ci.round(3), "\n")

# ---- 5. Calibration by risk group (Model A, unweighted) ----
groups = pd.qcut(p_A, 10, labels=False, duplicates="drop")
cal = pd.DataFrame({"group": groups, "predicted": p_A, "observed": y_val}) \
    .groupby("group").agg(n=("observed", "size"),
                          mean_predicted=("predicted", "mean"),
                          observed_rate=("observed", "mean"))
print("=== Model A calibration by predicted-risk decile (validation) ===")
print((cal * [1, 100, 100]).round(1).rename(
    columns={"mean_predicted": "predicted %", "observed_rate": "observed %"}))

Development 2017-2020: n = 4597, positive = 13.5%
Validation  2021-2023: n = 3587, positive = 10.6%

Fixed thresholds from 2017-2020: Model A = 0.106, Model B = 0.105, age rule = age >= 43

=== Validation 2021-2023, UNWEIGHTED (primary) ===
                 roc_auc  pr_auc  brier  cal_slope  obs_exp  sensitivity  specificity  precision  referred_pct  screen_per_case
Model A (final)    0.709   0.191  0.095      0.827    0.699        0.822        0.479      0.158        55.283            6.335
Model B            0.717   0.201  0.094      0.835    0.694        0.827        0.478      0.159        55.394            6.308
Age rule           0.700     NaN    NaN        NaN      NaN        0.832        0.445      0.151        58.433            6.612 

=== Validation 2021-2023, WEIGHTED, MEC exam weights (secondary) ===
                 roc_auc  pr_auc  brier  cal_slope  obs_exp  sensitivity  specificity  precision  referred_pct  screen_per_case
Model A (final)    0.730   0.173  0.078      0.8